# Adversarial Training Pipeline For Colab Free Tier

This notebook implements the build plan from `colab-build-plan.md` as a single standalone Colab notebook. It is designed for Google Colab Free and follows the reduced-scale proof-of-concept pipeline:

- public data download and preprocessing
- GPT-2 Small as the lightweight fake-text generator
- BioBERT as the detector
- BioGPT with LoRA as the adversarial rewrite agent
- round-based adversarial retraining with strict load/unload behavior
- Drive-based checkpoints, resume support, metrics logging, and plots

The notebook is organized by the exact phases in the plan so each section can be rerun independently after mounting Drive.


## Cell 1 — Install Dependencies

This cell installs the packages required by the Colab build plan. The notebook uses `accelerate` for training loops, `peft` for LoRA adapters, and standard Hugging Face auto-model classes everywhere.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'transformers>=4.38.0',
    'datasets>=2.18.0',
    'accelerate>=0.27.0',
    'scikit-learn>=1.3.0',
    'peft>=0.9.0',
    'bert-score>=0.3.13',
    'tqdm>=4.65.0',
    'pandas>=2.0.0',
    'numpy>=1.24.0',
    'matplotlib>=3.8.0',
    'sentencepiece>=0.1.99',
    'sacremoses>=0.1.1',
    'protobuf>=3.20.0',
    'huggingface_hub>=0.30.0',
])

print('Dependencies installed, including sacremoses for BioGPT tokenization and huggingface_hub for model uploads.')


## Cell 2 — Config Block

All configuration lives in the notebook in a single `CFG` dictionary. The values follow the Colab build plan, with a few extra helper paths added so the later cells can save checkpoints, plots, and round artifacts cleanly into Google Drive.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import html
import io
import json
import logging
import math
import os
import platform
import random
import re
import shutil
import subprocess
import unicodedata
import zipfile
from contextlib import nullcontext
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from accelerate import Accelerator
from datasets import load_dataset
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

try:
    from peft import LoraConfig, PeftModel, TaskType, get_peft_model
except Exception:
    LoraConfig = None
    PeftModel = None
    TaskType = None
    get_peft_model = None

CFG: dict[str, Any] = {
    'colab': {
        'drive_mount': '/content/drive',
        'save_dir': '/content/drive/MyDrive/deepfake_robustness',
        'checkpoint_every_round': True,
    },
    'data': {
        'pubmed_path': 'data/raw/pubmed_real.csv',
        'fake_path': 'data/raw/med_mmhl',
        'fake_csv_path': 'data/raw/fake_articles.csv',
        'processed_path': 'data/processed',
        'max_samples': 2000,
        'max_length': 256,
        'train_split': 0.70,
        'val_split': 0.15,
        'test_split': 0.15,
        'min_words': 50,
        'seed': 42,
        'text_columns': ['abstract', 'text', 'content', 'sentence', 'claim', 'article', 'body'],
        'title_column': 'title',
        'pubmed_dataset_name': 'slinusc/PubMedAbstractsSubset',
        'med_mmhl_repo': 'https://github.com/styxsys0927/Med-MMHL.git',
    },
    'generator': {
        'model_name': 'gpt2',
        'batch_size': 8,
        'max_new_tokens': 150,
        'finetune_epochs': 3,
        'lr': 2e-5,
        'checkpoint_path': 'models/generator/checkpoints',
        'temperature': 0.9,
        'top_p': 0.95,
    },
    'detector': {
        'model_name': 'dmis-lab/biobert-base-cased-v1.2',
        'num_labels': 2,
        'batch_size': 8,
        'max_length': 256,
        'epochs_per_round': 3,
        'lr': 2e-5,
        'weight_decay': 0.01,
        'warmup_ratio': 0.1,
        'checkpoint_dir': 'models/detector/checkpoints',
    },
    'agent': {
        'model_name': 'microsoft/biogpt',
        'evasion_threshold': 0.5,
        'high_conf_threshold': 0.8,
        'max_new_tokens': 128,
        'temperature': 0.9,
        'top_p': 0.95,
        'lora_r': 8,
        'lora_alpha': 16,
        'lora_dropout': 0.05,
        'finetune_epochs': 2,
        'batch_size': 4,
        'lr': 1e-4,
        'checkpoint_dir': 'models/agent/checkpoints',
        'gradient_checkpointing': True,
        'target_modules': ['q_proj', 'v_proj'],
    },
    'loop': {
        'num_rounds': 5,
        'fake_pool_size': 200,
        'hard_sample_top_k': 50,
        'save_checkpoint_every': 1,
        'round_data_dir': 'round_artifacts',
    },
    'runtime': {
        'metrics_csv': 'metrics_log.csv',
        'plots_dir': 'plots',
        'artifacts_dir': 'artifacts',
        'mixed_precision': 'fp16',
    },
}

CFG


## Cell 3 — Drive Mount And Helper Functions

This cell mounts Google Drive, creates the persistent working directories, and defines the helper functions requested in the plan:

- `save_checkpoint(obj, name, round_num)`
- `load_checkpoint(name, round_num)`
- `free_gpu(model)`
- `log_metrics(metrics_dict, round_num)`

It also adds a few notebook-specific helpers for path resolution, JSON saving, round resume detection, and checkpoint discovery.


In [ ]:
try:
    from google.colab import drive
    drive.mount(CFG['colab']['drive_mount'])
except Exception as exc:
    print(f'Drive mount skipped or unavailable: {exc}')

ROOT_DIR = Path(CFG['colab']['save_dir']).expanduser()
ROOT_DIR.mkdir(parents=True, exist_ok=True)
LOGGER = logging.getLogger('colab_adversarial_pipeline')
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(name)s | %(message)s', force=True)


def resolve_path(path: str | Path) -> Path:
    candidate = Path(path)
    if candidate.is_absolute():
        return candidate
    return ROOT_DIR / candidate


def ensure_dir(path: str | Path) -> Path:
    directory = resolve_path(path)
    directory.mkdir(parents=True, exist_ok=True)
    return directory


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def save_json(payload: dict, path: str | Path) -> Path:
    destination = resolve_path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open('w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2)
    return destination


def save_dataframe(frame: pd.DataFrame, path: str | Path) -> Path:
    destination = resolve_path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(destination, index=False)
    return destination


def save_checkpoint(obj: Any, name: str, round_num: Optional[int] = None) -> Path:
    checkpoint_dir = ensure_dir('artifacts/notebook_checkpoints')
    suffix = f'round_{round_num:02d}' if round_num is not None else 'latest'
    checkpoint_path = checkpoint_dir / f'{name}_{suffix}.pt'
    torch.save(obj, checkpoint_path)
    return checkpoint_path


def load_checkpoint(name: str, round_num: Optional[int] = None) -> Any:
    checkpoint_dir = ensure_dir('artifacts/notebook_checkpoints')
    suffix = f'round_{round_num:02d}' if round_num is not None else 'latest'
    checkpoint_path = checkpoint_dir / f'{name}_{suffix}.pt'
    if not checkpoint_path.exists():
        raise FileNotFoundError(checkpoint_path)
    return torch.load(checkpoint_path, map_location='cpu')


def free_gpu(model=None):
    try:
        del model
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return None


def normalize_metrics_row(row: dict[str, Any]) -> dict[str, Any]:
    normalized: dict[str, Any] = {}
    for key, value in row.items():
        if isinstance(value, Path):
            normalized[key] = str(value)
        elif isinstance(value, np.generic):
            normalized[key] = value.item()
        elif isinstance(value, (dict, list, tuple, set)):
            normalized[key] = json.dumps(value, ensure_ascii=True, sort_keys=True)
        else:
            normalized[key] = value
    return normalized


def load_metrics_frame_from_csv(metrics_path: Path) -> pd.DataFrame:
    import csv

    if not metrics_path.exists():
        return pd.DataFrame()
    with metrics_path.open('r', encoding='utf-8', newline='') as handle:
        rows = list(csv.reader(handle))
    if not rows:
        return pd.DataFrame()
    header = list(rows[0])
    max_fields = max(len(row) for row in rows if row)
    if len(header) < max_fields:
        header.extend([f'unnamed_extra_{index}' for index in range(len(header), max_fields)])
    records: List[dict[str, Any]] = []
    for row in rows[1:]:
        if not row:
            continue
        padded = list(row) + [''] * (len(header) - len(row))
        records.append(dict(zip(header, padded[:len(header)])))
    return pd.DataFrame(records)


def load_metrics_frame_from_artifacts() -> pd.DataFrame:
    records: List[dict[str, Any]] = []
    round0_metrics_path = resolve_path(Path(CFG['runtime']['artifacts_dir']) / 'round_00_metrics.json')
    if round0_metrics_path.exists():
        with round0_metrics_path.open('r', encoding='utf-8') as handle:
            payload = json.load(handle)
        records.append(normalize_metrics_row(payload))
    round_root = resolve_path(CFG['loop']['round_data_dir'])
    if round_root.exists():
        for round_dir in sorted(path for path in round_root.glob('round_*') if path.is_dir()):
            try:
                round_num = int(round_dir.name.split('_')[-1])
            except ValueError:
                continue
            metrics_json = round_dir / 'round_metrics.json'
            error_json = round_dir / 'round_error.json'
            if metrics_json.exists():
                with metrics_json.open('r', encoding='utf-8') as handle:
                    payload = json.load(handle)
                records.append(normalize_metrics_row({'round': round_num, **payload}))
            elif error_json.exists():
                with error_json.open('r', encoding='utf-8') as handle:
                    payload = json.load(handle)
                records.append(normalize_metrics_row({'round': round_num, **payload}))
    return pd.DataFrame(records)


def load_metrics_frame() -> pd.DataFrame:
    metrics_path = resolve_path(CFG['runtime']['metrics_csv'])
    frames = []
    csv_frame = load_metrics_frame_from_csv(metrics_path)
    if not csv_frame.empty:
        frames.append(csv_frame)
    artifact_frame = load_metrics_frame_from_artifacts()
    if not artifact_frame.empty:
        frames.append(artifact_frame)
    if not frames:
        return pd.DataFrame()
    frame = pd.concat(frames, ignore_index=True, sort=False)
    if 'round' not in frame.columns:
        return frame
    frame['round'] = pd.to_numeric(frame['round'], errors='coerce')
    frame = frame.dropna(subset=['round']).copy()
    frame['round'] = frame['round'].astype(int)
    frame = frame.drop_duplicates(subset=['round'], keep='last').sort_values('round').reset_index(drop=True)
    ordered_columns = ['round'] + [column for column in frame.columns if column != 'round']
    return frame[ordered_columns]


def log_metrics(metrics_dict: dict, round_num: int) -> Path:
    metrics_path = resolve_path(CFG['runtime']['metrics_csv'])
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    existing = load_metrics_frame()
    row = pd.DataFrame([normalize_metrics_row({'round': round_num, **metrics_dict})])
    frame = pd.concat([existing, row], ignore_index=True, sort=False)
    frame['round'] = pd.to_numeric(frame['round'], errors='coerce')
    frame = frame.dropna(subset=['round']).copy()
    frame['round'] = frame['round'].astype(int)
    frame = frame.drop_duplicates(subset=['round'], keep='last').sort_values('round').reset_index(drop=True)
    ordered_columns = ['round'] + [column for column in frame.columns if column != 'round']
    frame = frame[ordered_columns]
    frame.to_csv(metrics_path, index=False)
    return metrics_path


def get_device() -> torch.device:
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def latest_completed_round() -> int:
    frame = load_metrics_frame()
    if frame.empty or 'status' not in frame.columns:
        return 0
    completed = frame[frame['status'] == 'completed']
    if completed.empty:
        return 0
    return int(completed['round'].max())


def detector_round_dir(round_num: Optional[int] = None) -> Path:
    base_dir = ensure_dir(CFG['detector']['checkpoint_dir'])
    return base_dir / ('baseline' if round_num in (None, 0) else f'round_{round_num:02d}')


def agent_round_dir(round_num: int) -> Path:
    return ensure_dir(CFG['agent']['checkpoint_dir']) / f'round_{round_num:02d}'


def generator_checkpoint_dir() -> Path:
    return ensure_dir(CFG['generator']['checkpoint_path']) / 'gpt2_generator'


def find_latest_detector_checkpoint(max_round: Optional[int] = None) -> Path:
    base = detector_round_dir(0)
    if max_round is None:
        max_round = CFG['loop']['num_rounds']
    available = [base]
    for round_num in range(1, max_round + 1):
        candidate = detector_round_dir(round_num)
        if candidate.exists() and any(candidate.iterdir()):
            available.append(candidate)
    return available[-1]


def find_latest_agent_adapter(max_round: Optional[int] = None) -> Optional[Path]:
    if max_round is None:
        max_round = CFG['loop']['num_rounds']
    available: List[Path] = []
    for round_num in range(1, max_round + 1):
        candidate = agent_round_dir(round_num)
        if candidate.exists() and (candidate / 'adapter_config.json').exists():
            available.append(candidate)
    return available[-1] if available else None

set_seed(CFG['data']['seed'])
ensure_dir(CFG['data']['processed_path'])
ensure_dir(CFG['runtime']['plots_dir'])
ensure_dir(CFG['runtime']['artifacts_dir'])
print(f'Root save dir: {ROOT_DIR}')
print(f'Device: {get_device()}')


## Cell 4 — Download Data

This cell downloads the real PubMed subset, clones the public Med-MMHL repository if it is not present yet, extracts a usable text column from the fake-data files, and saves both classes as CSV files in Drive-backed storage.


In [ ]:
HTML_TAG_PATTERN = re.compile(r'<[^>]+>')
WHITESPACE_PATTERN = re.compile(r'\s+')
SUPPORTED_SUFFIXES = {'.csv', '.json', '.jsonl', '.txt'}


def discover_text_column(frame: pd.DataFrame, candidates: Sequence[str]) -> str:
    for column in candidates:
        if column in frame.columns:
            return column
    raise ValueError(f'No text column found in columns={list(frame.columns)}')


def load_file_like(name: str, raw_bytes: bytes) -> pd.DataFrame:
    suffix = Path(name).suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(io.BytesIO(raw_bytes))
    if suffix == '.json':
        return pd.read_json(io.BytesIO(raw_bytes))
    if suffix == '.jsonl':
        return pd.read_json(io.BytesIO(raw_bytes), lines=True)
    if suffix == '.txt':
        text = raw_bytes.decode('utf-8', errors='ignore')
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        return pd.DataFrame({'text': lines})
    raise ValueError(f'Unsupported source format: {name}')


def load_source(path: Path) -> pd.DataFrame:
    if path.is_dir():
        frames: List[pd.DataFrame] = []
        for child in sorted(path.iterdir()):
            if child.is_dir() or child.suffix.lower() in SUPPORTED_SUFFIXES | {'.zip'}:
                frames.append(load_source(child))
        if not frames:
            raise FileNotFoundError(f'No supported source files found under {path}')
        return pd.concat(frames, ignore_index=True)
    if path.suffix.lower() == '.zip':
        frames: List[pd.DataFrame] = []
        with zipfile.ZipFile(path, 'r') as archive:
            for member in sorted(archive.namelist()):
                if Path(member).suffix.lower() in SUPPORTED_SUFFIXES and not member.endswith('/'):
                    with archive.open(member) as handle:
                        frames.append(load_file_like(member, handle.read()))
        if not frames:
            raise FileNotFoundError(f'No supported files found inside archive: {path}')
        return pd.concat(frames, ignore_index=True)
    return load_file_like(path.name, path.read_bytes())


def find_uploaded_component(name: str, search_roots: Sequence[Path]) -> Optional[Path]:
    for root in search_roots:
        direct = root / name
        if direct.exists():
            return direct
        matches = sorted(candidate for candidate in root.rglob(name) if candidate.name == name)
        if matches:
            return matches[0]
    return None


def copy_uploaded_component(source: Path, destination_root: Path) -> None:
    target = destination_root / source.name
    if target.exists():
        if target.is_dir():
            shutil.rmtree(target)
        else:
            target.unlink()
    if source.is_dir():
        shutil.copytree(source, target)
    else:
        shutil.copy2(source, target)


pubmed_csv_path = resolve_path(CFG['data']['pubmed_path'])
pubmed_csv_path.parent.mkdir(parents=True, exist_ok=True)
if not pubmed_csv_path.exists():
    streamed_pubmed = load_dataset(CFG['data']['pubmed_dataset_name'], split='train', streaming=True)
    pubmed_rows: List[Dict[str, Any]] = []
    for item in streamed_pubmed:
        title = item.get('title')
        abstract = item.get('abstract')
        if not title or not abstract:
            continue
        pubmed_rows.append({'title': str(title), 'text': str(abstract), 'label': 0})
        if len(pubmed_rows) >= CFG['data']['max_samples']:
            break
    pubmed_frame = pd.DataFrame(pubmed_rows)
    if pubmed_frame.empty:
        raise RuntimeError('PubMed download returned no usable rows.')
    pubmed_frame.to_csv(pubmed_csv_path, index=False)
else:
    pubmed_frame = pd.read_csv(pubmed_csv_path)

med_mmhl_dir = resolve_path(CFG['data']['fake_path'])
uploaded_fake_items = ['fakenews_article', 'sentence']
uploaded_sources = [
    found_path
    for found_path in (find_uploaded_component(name, [Path('/content')]) for name in uploaded_fake_items)
    if found_path is not None
]
if uploaded_sources:
    if med_mmhl_dir.exists():
        shutil.rmtree(med_mmhl_dir)
    med_mmhl_dir.mkdir(parents=True, exist_ok=True)
    for source in uploaded_sources:
        copy_uploaded_component(source, med_mmhl_dir)
    fake_source_label = 'uploaded folders from /content'
elif not med_mmhl_dir.exists() or not any(med_mmhl_dir.iterdir()):
    med_mmhl_dir.parent.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.check_call(['git', 'clone', CFG['data']['med_mmhl_repo'], str(med_mmhl_dir)])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Failed to fetch the fake dataset source. Upload your `fakenews_article` and `sentence` folders '
            f'to Colab or verify that this repo is reachable: {CFG["data"]["med_mmhl_repo"]}'
        ) from exc
    fake_source_label = CFG['data']['med_mmhl_repo']
else:
    fake_source_label = str(med_mmhl_dir)

fake_raw_frame = load_source(med_mmhl_dir)
fake_text_column = discover_text_column(fake_raw_frame, CFG['data']['text_columns'])
fake_standardized = pd.DataFrame({
    'title': fake_raw_frame[CFG['data']['title_column']].astype(str) if CFG['data']['title_column'] in fake_raw_frame.columns else '',
    'text': fake_raw_frame[fake_text_column].astype(str),
    'label': 1,
})
fake_csv_path = save_dataframe(fake_standardized, CFG['data']['fake_csv_path'])

print(f'PubMed rows: {len(pubmed_frame)}')
print(f'Fake rows: {len(fake_standardized)}')
print(f'Fake source: {fake_source_label}')
print(f'Real CSV: {pubmed_csv_path}')
print(f'Fake CSV: {fake_csv_path}')


## Cell 5 — Prepare Data

This cell merges the real and fake CSV files, cleans the text, caps the sample counts, deduplicates, truncates to the reduced Colab length, creates stratified train/validation/test splits, saves them to Drive, and prints the resulting class distributions.


In [ ]:
def normalize_text(text: str) -> str:
    cleaned = html.unescape(str(text))
    cleaned = HTML_TAG_PATTERN.sub(' ', cleaned)
    cleaned = unicodedata.normalize('NFKC', cleaned)
    cleaned = WHITESPACE_PATTERN.sub(' ', cleaned).strip()
    return cleaned


def text_hash(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def truncate_text(text: str, tokenizer: AutoTokenizer, max_length: int) -> str:
    token_ids = tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=max_length)
    return tokenizer.decode(token_ids, skip_special_tokens=True)


def split_frame(frame: pd.DataFrame, train_ratio: float, val_ratio: float, test_ratio: float, seed: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_frame, temp_frame = train_test_split(frame, test_size=(1.0 - train_ratio), random_state=seed, stratify=frame['label'])
    relative_test_ratio = test_ratio / (val_ratio + test_ratio)
    val_frame, test_frame = train_test_split(temp_frame, test_size=relative_test_ratio, random_state=seed, stratify=temp_frame['label'])
    return train_frame.reset_index(drop=True), val_frame.reset_index(drop=True), test_frame.reset_index(drop=True)


real_frame = pd.read_csv(resolve_path(CFG['data']['pubmed_path']))[['title', 'text', 'label']].copy()
fake_frame = pd.read_csv(resolve_path(CFG['data']['fake_csv_path']))[['title', 'text', 'label']].copy()

if len(real_frame) > CFG['data']['max_samples']:
    real_frame = real_frame.sample(CFG['data']['max_samples'], random_state=CFG['data']['seed']).reset_index(drop=True)
if len(fake_frame) > CFG['data']['max_samples']:
    fake_frame = fake_frame.sample(CFG['data']['max_samples'], random_state=CFG['data']['seed']).reset_index(drop=True)

biobert_tokenizer_for_truncation = AutoTokenizer.from_pretrained(CFG['detector']['model_name'])
merged_frame = pd.concat([real_frame, fake_frame], ignore_index=True)
merged_frame['text'] = merged_frame['text'].fillna('').map(normalize_text)
merged_frame = merged_frame[merged_frame['text'].astype(bool)].copy()
merged_frame['word_count'] = merged_frame['text'].astype(str).str.split().str.len()
merged_frame = merged_frame[merged_frame['word_count'] >= CFG['data']['min_words']].copy()
merged_frame['text'] = merged_frame['text'].map(lambda value: truncate_text(value, biobert_tokenizer_for_truncation, CFG['data']['max_length']))
merged_frame['text_hash'] = merged_frame['text'].map(text_hash)
merged_frame = merged_frame.drop_duplicates(subset='text_hash').drop(columns=['word_count', 'text_hash']).reset_index(drop=True)

train_frame, val_frame, test_frame = split_frame(
    merged_frame,
    train_ratio=CFG['data']['train_split'],
    val_ratio=CFG['data']['val_split'],
    test_ratio=CFG['data']['test_split'],
    seed=CFG['data']['seed'],
)

train_csv = save_dataframe(train_frame, Path(CFG['data']['processed_path']) / 'train.csv')
val_csv = save_dataframe(val_frame, Path(CFG['data']['processed_path']) / 'val.csv')
test_csv = save_dataframe(test_frame, Path(CFG['data']['processed_path']) / 'test.csv')

for split_name, frame in [('train', train_frame), ('val', val_frame), ('test', test_frame)]:
    distribution = frame['label'].value_counts().sort_index().to_dict()
    print(split_name, 'rows =', len(frame), 'distribution =', distribution)

save_json(
    {
        'train_rows': len(train_frame),
        'val_rows': len(val_frame),
        'test_rows': len(test_frame),
        'train_path': str(train_csv),
        'val_path': str(val_csv),
        'test_path': str(test_csv),
    },
    Path(CFG['runtime']['artifacts_dir']) / 'data_summary.json',
)


## Cell 6 — Fine-Tune GPT-2 Generator

This cell fine-tunes GPT-2 Small as the lightweight fake-text generator. It uses only the fake rows from `train.csv`, prepends the `[FAKE] ` prefix during training, and saves the generator checkpoint to Drive.


In [ ]:
class CausalTextDataset(Dataset):
    def __init__(self, texts: Sequence[str], tokenizer: AutoTokenizer, max_length: int, prefix: str = '') -> None:
        self.texts = [f'{prefix}{text}' for text in texts]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> dict:
        encoding = self.tokenizer(
            self.texts[index],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt',
        )
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': input_ids.clone(),
        }


def train_generator(cfg: dict, train_csv_path: str | Path, output_dir: str | Path) -> dict:
    accelerator = Accelerator(mixed_precision=cfg['runtime']['mixed_precision'] if torch.cuda.is_available() else 'no')
    generator_cfg = cfg['generator']
    data_cfg = cfg['data']

    fake_train_frame = pd.read_csv(resolve_path(train_csv_path))
    fake_train_texts = fake_train_frame[fake_train_frame['label'] == 1]['text'].astype(str).tolist()
    tokenizer = AutoTokenizer.from_pretrained(generator_cfg['model_name'])
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(generator_cfg['model_name'])

    dataset = CausalTextDataset(fake_train_texts, tokenizer, data_cfg['max_length'], prefix='[FAKE] ')
    dataloader = DataLoader(dataset, batch_size=generator_cfg['batch_size'], shuffle=True)
    optimizer = AdamW(model.parameters(), lr=generator_cfg['lr'])
    total_steps = max(len(dataloader) * generator_cfg['finetune_epochs'], 1)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    model, optimizer, dataloader, scheduler = accelerator.prepare(model, optimizer, dataloader, scheduler)
    history: List[dict] = []
    model.train()
    for epoch in range(generator_cfg['finetune_epochs']):
        losses: List[float] = []
        progress = tqdm(dataloader, desc=f'gpt2 generator epoch {epoch + 1}', leave=False)
        for batch in progress:
            outputs = model(**batch)
            loss = outputs.loss
            accelerator.backward(loss)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            losses.append(float(loss.detach().cpu().item()))
            progress.set_postfix(loss=f'{losses[-1]:.4f}')
        history.append({'epoch': epoch + 1, 'train_loss': float(np.mean(losses) if losses else math.nan)})

    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    output_dir = ensure_dir(output_dir)
    if accelerator.is_main_process:
        unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
        tokenizer.save_pretrained(output_dir)
        save_json({'history': history}, Path(output_dir) / 'training_summary.json')

    del unwrapped_model
    del model
    del optimizer
    del dataloader
    del scheduler
    del accelerator
    free_gpu()
    return {'output_dir': str(output_dir), 'history': history}


generator_ckpt_dir = generator_checkpoint_dir()
generator_summary = train_generator(CFG, train_csv, generator_ckpt_dir)
generator_summary


## Cell 7 — Generator Inference Function

This cell defines `generate_fake_batch(n, model, tokenizer)` and runs a small smoke test. It loads the saved GPT-2 checkpoint, generates a few samples from the `[FAKE] ` prompt, prints them, and unloads the model afterwards.


In [ ]:
def load_generator_model(checkpoint_dir: str | Path):
    device = get_device()
    checkpoint_dir = resolve_path(checkpoint_dir)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(checkpoint_dir)
    model.to(device)
    model.eval()
    return model, tokenizer, device


def generate_fake_batch(n: int, model, tokenizer, batch_size: Optional[int] = None) -> List[str]:
    batch_size = batch_size or CFG['generator']['batch_size']
    results: List[str] = []
    prompt = '[FAKE] '
    while len(results) < n:
        current_batch = min(batch_size, n - len(results))
        encoded = tokenizer(
            [prompt] * current_batch,
            padding=True,
            truncation=True,
            max_length=16,
            return_tensors='pt',
        )
        encoded = {key: value.to(model.device) for key, value in encoded.items()}
        with torch.inference_mode():
            outputs = model.generate(
                **encoded,
                do_sample=True,
                temperature=CFG['generator']['temperature'],
                top_p=CFG['generator']['top_p'],
                max_new_tokens=CFG['generator']['max_new_tokens'],
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        prompt_lengths = encoded['attention_mask'].sum(dim=1).tolist()
        for row_idx, generated_ids in enumerate(outputs):
            new_ids = generated_ids[int(prompt_lengths[row_idx]):]
            text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
            if text:
                results.append(text)
    return results[:n]


generator_model, generator_tokenizer, _ = load_generator_model(generator_ckpt_dir)
smoke_samples = generate_fake_batch(5, generator_model, generator_tokenizer)
for index, sample in enumerate(smoke_samples, start=1):
    print(f'Sample {index}:\n{sample}\n')

generator_model = free_gpu(generator_model)


## Cell 8 — Fine-Tune BioBERT Detector

This cell trains the baseline BioBERT detector on `train.csv` and evaluates on `val.csv` at the end of each epoch. It saves the best checkpoint by validation F1 into Drive.


In [ ]:
class DetectorDataset(Dataset):
    def __init__(self, texts: Sequence[str], labels: Sequence[int], tokenizer: AutoTokenizer, max_length: int) -> None:
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> dict:
        encoding = self.tokenizer(
            self.texts[index],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt',
        )
        item = {key: value.squeeze(0) for key, value in encoding.items()}
        item['labels'] = torch.tensor(self.labels[index], dtype=torch.long)
        return item


def compute_auc_f1_from_probs(labels: Sequence[int], probabilities: Sequence[float], threshold: float = 0.5) -> dict:
    label_array = np.asarray(labels)
    prob_array = np.asarray(probabilities, dtype=float)
    predictions = (prob_array >= threshold).astype(int)
    auc = float(roc_auc_score(label_array, prob_array)) if len(np.unique(label_array)) > 1 else math.nan
    f1 = float(f1_score(label_array, predictions, zero_division=0))
    return {'auc': auc, 'f1': f1}


def train_detector_model(cfg: dict, train_csv_path: str | Path, val_csv_path: str | Path, output_dir: str | Path, init_checkpoint: Optional[str | Path] = None) -> dict:
    accelerator = Accelerator(mixed_precision=cfg['runtime']['mixed_precision'] if torch.cuda.is_available() else 'no')
    detector_cfg = cfg['detector']

    train_frame = pd.read_csv(resolve_path(train_csv_path))
    val_frame = pd.read_csv(resolve_path(val_csv_path))
    model_source = str(resolve_path(init_checkpoint)) if init_checkpoint else detector_cfg['model_name']
    tokenizer = AutoTokenizer.from_pretrained(model_source)
    model = AutoModelForSequenceClassification.from_pretrained(model_source, num_labels=detector_cfg['num_labels'])
    if hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()
        model.config.use_cache = False

    train_dataset = DetectorDataset(train_frame['text'].astype(str).tolist(), train_frame['label'].astype(int).tolist(), tokenizer, detector_cfg['max_length'])
    val_dataset = DetectorDataset(val_frame['text'].astype(str).tolist(), val_frame['label'].astype(int).tolist(), tokenizer, detector_cfg['max_length'])
    train_loader = DataLoader(train_dataset, batch_size=detector_cfg['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=detector_cfg['batch_size'], shuffle=False)

    optimizer = AdamW(model.parameters(), lr=detector_cfg['lr'], weight_decay=detector_cfg['weight_decay'])
    total_steps = max(len(train_loader) * detector_cfg['epochs_per_round'], 1)
    warmup_steps = int(total_steps * detector_cfg['warmup_ratio'])
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    model, optimizer, train_loader, val_loader, scheduler = accelerator.prepare(model, optimizer, train_loader, val_loader, scheduler)
    best_f1 = float('-inf')
    best_metrics: dict[str, Any] = {}
    history: List[dict] = []

    for epoch in range(detector_cfg['epochs_per_round']):
        model.train()
        train_losses: List[float] = []
        progress = tqdm(train_loader, desc=f'biobert epoch {epoch + 1}', leave=False)
        for batch in progress:
            outputs = model(**batch)
            loss = outputs.loss
            accelerator.backward(loss)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            train_losses.append(float(loss.detach().cpu().item()))
            progress.set_postfix(loss=f'{train_losses[-1]:.4f}')

        model.eval()
        val_labels: List[int] = []
        val_probs: List[float] = []
        with torch.inference_mode():
            for batch in val_loader:
                outputs = model(**batch)
                probs = torch.softmax(outputs.logits, dim=-1)[:, 1]
                gathered_probs = accelerator.gather_for_metrics(probs)
                gathered_labels = accelerator.gather_for_metrics(batch['labels'])
                val_probs.extend(gathered_probs.detach().cpu().tolist())
                val_labels.extend(gathered_labels.detach().cpu().tolist())

        metrics = compute_auc_f1_from_probs(val_labels, val_probs)
        epoch_record = {'epoch': epoch + 1, 'train_loss': float(np.mean(train_losses) if train_losses else math.nan), **metrics}
        history.append(epoch_record)

        if accelerator.is_main_process and metrics['f1'] >= best_f1:
            best_f1 = metrics['f1']
            best_metrics = epoch_record
            output_dir = ensure_dir(output_dir)
            accelerator.unwrap_model(model).save_pretrained(output_dir, save_function=accelerator.save)
            tokenizer.save_pretrained(output_dir)
            save_json({'history': history, 'best_metrics': best_metrics}, Path(output_dir) / 'training_summary.json')

    accelerator.wait_for_everyone()
    del model
    del optimizer
    del train_loader
    del val_loader
    del scheduler
    del accelerator
    free_gpu()
    return {'output_dir': str(output_dir), 'best_metrics': best_metrics, 'history': history}


baseline_detector_dir = detector_round_dir(0)
baseline_summary = train_detector_model(CFG, train_csv, val_csv, baseline_detector_dir)
baseline_summary


## Cell 9 — Scorer Function And Round 0 Baseline

This cell defines the detector scoring helpers, evaluates the saved baseline checkpoint on `test.csv`, logs round 0 AUC and F1 into `metrics_log.csv`, and unloads the model.


In [ ]:
def load_detector_model(checkpoint_dir: str | Path):
    device = get_device()
    checkpoint_dir = resolve_path(checkpoint_dir)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
    model.to(device)
    model.eval()
    return model, tokenizer


def score_batch(texts: Sequence[str], model, tokenizer, batch_size: Optional[int] = None) -> List[float]:
    batch_size = batch_size or CFG['detector']['batch_size']
    results: List[float] = []
    for start in range(0, len(texts), batch_size):
        batch_texts = list(texts[start : start + batch_size])
        encoded = tokenizer(
            batch_texts,
            truncation=True,
            padding='max_length',
            max_length=CFG['detector']['max_length'],
            return_tensors='pt',
        )
        encoded = {key: value.to(model.device) for key, value in encoded.items()}
        with torch.inference_mode():
            outputs = model(**encoded)
            probs = torch.softmax(outputs.logits, dim=-1)[:, 1]
        results.extend(probs.detach().cpu().tolist())
    return results


def evaluate_detector_on_csv(csv_path: str | Path, checkpoint_dir: str | Path) -> tuple[dict, pd.DataFrame]:
    frame = pd.read_csv(resolve_path(csv_path))
    model, tokenizer = load_detector_model(checkpoint_dir)
    probabilities = score_batch(frame['text'].astype(str).tolist(), model, tokenizer)
    metrics = compute_auc_f1_from_probs(frame['label'].astype(int).tolist(), probabilities)
    prediction_frame = frame[['title', 'text', 'label']].copy()
    prediction_frame['fake_probability'] = probabilities
    prediction_frame['predicted_label'] = (prediction_frame['fake_probability'] >= CFG['agent']['evasion_threshold']).astype(int)
    model = free_gpu(model)
    return metrics, prediction_frame


round0_metrics, round0_predictions = evaluate_detector_on_csv(test_csv, baseline_detector_dir)
round0_metrics.update({'evasion_rate': math.nan, 'robustness_delta': 0.0, 'status': 'completed'})
log_metrics(round0_metrics, round_num=0)
save_json({'round': 0, **round0_metrics}, Path(CFG['runtime']['artifacts_dir']) / 'round_00_metrics.json')
save_dataframe(round0_predictions, Path(CFG['runtime']['artifacts_dir']) / 'round_00_predictions.csv')
round0_metrics


## Cell 10 — Prompt Templates

This cell defines the rewrite and generation prompts for the BioGPT adversarial agent.


In [ ]:
REWRITE_PROMPT = """System: You are a biomedical text writer. Rewrite the following abstract to make it harder for a machine-learning detector to classify as synthetic while preserving its scientific meaning and core claims. Do not add new facts.

Original abstract:
{fake_abstract}

Rewritten abstract:
"""

GENERATE_PROMPT = """System: Write a plausible but fictitious biomedical abstract about the following topic. The style should resemble a scientific publication.

Topic: {topic}

Abstract:
"""

REWRITE_PROMPT, GENERATE_PROMPT


## Cell 11 — BioGPT + LoRA Setup And Agent Functions

This cell loads BioGPT through the Hugging Face auto classes, attaches LoRA adapters with `peft`, defines `rewrite_batch`, and defines `finetune_on_successes` for round-by-round adapter updates. Only adapter weights are saved to Drive.


In [ ]:
class PromptResponseDataset(Dataset):
    def __init__(self, prompt_response_pairs: Sequence[tuple[str, str]], tokenizer: AutoTokenizer, max_length: int) -> None:
        self.records = []
        for prompt, response in prompt_response_pairs:
            prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
            response_ids = tokenizer(response, add_special_tokens=False)['input_ids']
            input_ids = prompt_ids + response_ids + [tokenizer.eos_token_id]
            input_ids = input_ids[:max_length]
            attention_mask = [1] * len(input_ids)
            labels = [-100] * min(len(prompt_ids), len(input_ids))
            labels.extend(input_ids[len(labels):])
            pad_len = max_length - len(input_ids)
            if pad_len > 0:
                input_ids += [tokenizer.pad_token_id] * pad_len
                attention_mask += [0] * pad_len
                labels += [-100] * pad_len
            self.records.append({
                'input_ids': torch.tensor(input_ids, dtype=torch.long),
                'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
                'labels': torch.tensor(labels, dtype=torch.long),
            })

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict:
        return self.records[index]


def load_agent_model(adapter_dir: Optional[str | Path] = None, for_training: bool = False):
    if LoraConfig is None or get_peft_model is None:
        raise ImportError('peft is required for the BioGPT agent cell.')
    device = get_device()
    try:
        tokenizer = AutoTokenizer.from_pretrained(CFG['agent']['model_name'])
    except Exception as exc:
        if 'sacremoses' in str(exc).lower():
            raise ImportError(
                'BioGPT tokenization requires `sacremoses`. Rerun the dependency-install cell, '
                'or run `pip install sacremoses`, then rerun the BioGPT cells.'
            ) from exc
        raise
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    base_model = AutoModelForCausalLM.from_pretrained(CFG['agent']['model_name'])

    if for_training:
        if CFG['agent']['gradient_checkpointing'] and hasattr(base_model, 'gradient_checkpointing_enable'):
            base_model.gradient_checkpointing_enable()
            base_model.config.use_cache = False
        if adapter_dir is not None and (resolve_path(adapter_dir) / 'adapter_config.json').exists():
            model = PeftModel.from_pretrained(base_model, resolve_path(adapter_dir), is_trainable=True)
        else:
            lora_config = LoraConfig(
                r=CFG['agent']['lora_r'],
                lora_alpha=CFG['agent']['lora_alpha'],
                lora_dropout=CFG['agent']['lora_dropout'],
                target_modules=CFG['agent']['target_modules'],
                bias='none',
                task_type=TaskType.CAUSAL_LM,
            )
            model = get_peft_model(base_model, lora_config)
    else:
        if adapter_dir is not None and (resolve_path(adapter_dir) / 'adapter_config.json').exists():
            model = PeftModel.from_pretrained(base_model, resolve_path(adapter_dir))
        else:
            model = base_model

    model.to(device)
    model.eval()
    return model, tokenizer


def rewrite_batch(texts: Sequence[str], scores: Sequence[float], model, tokenizer) -> List[str]:
    selected_indices = [index for index, score in enumerate(scores) if score > CFG['agent']['high_conf_threshold']]
    rewritten = list(texts)
    if not selected_indices:
        return rewritten
    prompts = [REWRITE_PROMPT.format(fake_abstract=texts[index]) for index in selected_indices]
    batch_size = CFG['agent']['batch_size']
    generated_texts: List[str] = []
    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start : start + batch_size]
        encoded = tokenizer(
            batch_prompts,
            padding=True,
            truncation=True,
            max_length=CFG['detector']['max_length'],
            return_tensors='pt',
        )
        encoded = {key: value.to(model.device) for key, value in encoded.items()}
        with torch.inference_mode():
            outputs = model.generate(
                **encoded,
                do_sample=True,
                max_new_tokens=CFG['agent']['max_new_tokens'],
                temperature=CFG['agent']['temperature'],
                top_p=CFG['agent']['top_p'],
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        prompt_lengths = encoded['attention_mask'].sum(dim=1).tolist()
        for row_idx, generated_ids in enumerate(outputs):
            new_ids = generated_ids[int(prompt_lengths[row_idx]):]
            generated_texts.append(tokenizer.decode(new_ids, skip_special_tokens=True).strip())
    for idx, generated_text in zip(selected_indices, generated_texts):
        rewritten[idx] = generated_text if generated_text else texts[idx]
    return rewritten


def finetune_on_successes(success_pairs: Sequence[dict], model, tokenizer, output_dir: str | Path) -> dict:
    if not success_pairs:
        return {'output_dir': str(resolve_path(output_dir)), 'examples': 0}
    accelerator = Accelerator(mixed_precision=CFG['runtime']['mixed_precision'] if torch.cuda.is_available() else 'no')
    prompt_response_pairs = [
        (REWRITE_PROMPT.format(fake_abstract=item['original_text']), item['rewritten_text'])
        for item in success_pairs
    ]
    dataset = PromptResponseDataset(prompt_response_pairs, tokenizer, CFG['detector']['max_length'])
    dataloader = DataLoader(dataset, batch_size=CFG['agent']['batch_size'], shuffle=True)
    optimizer = AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=CFG['agent']['lr'])
    total_steps = max(len(dataloader) * CFG['agent']['finetune_epochs'], 1)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    model.train()
    model, optimizer, dataloader, scheduler = accelerator.prepare(model, optimizer, dataloader, scheduler)
    history: List[dict] = []
    for epoch in range(CFG['agent']['finetune_epochs']):
        losses: List[float] = []
        progress = tqdm(dataloader, desc=f'biogpt lora epoch {epoch + 1}', leave=False)
        for batch in progress:
            outputs = model(**batch)
            loss = outputs.loss
            accelerator.backward(loss)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            losses.append(float(loss.detach().cpu().item()))
            progress.set_postfix(loss=f'{losses[-1]:.4f}')
        history.append({'epoch': epoch + 1, 'train_loss': float(np.mean(losses) if losses else math.nan)})

    accelerator.wait_for_everyone()
    output_dir = ensure_dir(output_dir)
    if accelerator.is_main_process:
        unwrapped_model = accelerator.unwrap_model(model)
        unwrapped_model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
        save_json({'history': history, 'examples': len(success_pairs)}, Path(output_dir) / 'training_summary.json')

    del model
    del optimizer
    del dataloader
    del scheduler
    del accelerator
    free_gpu()
    return {'output_dir': str(output_dir), 'examples': len(success_pairs), 'history': history}


## Cell 12 — Adversarial Loop

This cell implements the full round-based loop from the build plan with strict load/unload behavior, per-round checkpoints, Drive logging, and resume logic. Each round is wrapped in `try/except` so an error can be logged without losing earlier progress.


In [ ]:
def compute_round_auc_f1(labels: Sequence[int], probabilities: Sequence[float]) -> dict:
    return compute_auc_f1_from_probs(labels, probabilities, threshold=CFG['agent']['evasion_threshold'])


def run_test_evaluation(checkpoint_dir: str | Path) -> tuple[dict, pd.DataFrame]:
    frame = pd.read_csv(resolve_path(test_csv))
    model, tokenizer = load_detector_model(checkpoint_dir)
    probabilities = score_batch(frame['text'].astype(str).tolist(), model, tokenizer)
    metrics = compute_round_auc_f1(frame['label'].astype(int).tolist(), probabilities)
    prediction_frame = frame[['title', 'text', 'label']].copy()
    prediction_frame['fake_probability'] = probabilities
    prediction_frame['predicted_label'] = (prediction_frame['fake_probability'] >= CFG['agent']['evasion_threshold']).astype(int)
    model = free_gpu(model)
    return metrics, prediction_frame


def compute_evasion_rate_quick(scores: Sequence[float], threshold: float) -> float:
    if not scores:
        return 0.0
    return float(sum(score < threshold for score in scores) / len(scores))


base_train_frame = pd.read_csv(resolve_path(train_csv))
baseline_auc = round0_metrics['auc']
start_round = latest_completed_round() + 1
print(f'Resuming from round {start_round}')

for round_num in range(start_round, CFG['loop']['num_rounds'] + 1):
    if round_num <= 0:
        continue
    round_dir = ensure_dir(Path(CFG['loop']['round_data_dir']) / f'round_{round_num:02d}')
    try:
        print(f'\n===== ROUND {round_num} =====')
        detector_ckpt = find_latest_detector_checkpoint(max_round=round_num - 1)
        agent_adapter = find_latest_agent_adapter(max_round=round_num - 1)

        # STEP 1 — Generate fake pool
        generator_model, generator_tokenizer, _ = load_generator_model(generator_ckpt_dir)
        fake_pool = generate_fake_batch(CFG['loop']['fake_pool_size'], generator_model, generator_tokenizer)
        generator_model = free_gpu(generator_model)
        save_dataframe(pd.DataFrame({'generated_text': fake_pool}), round_dir / 'fake_pool.csv')

        # STEP 2 — Score fake pool with the latest detector
        detector_model, detector_tokenizer = load_detector_model(detector_ckpt)
        fake_scores = score_batch(fake_pool, detector_model, detector_tokenizer)
        ranking = sorted(range(len(fake_pool)), key=lambda index: fake_scores[index], reverse=True)
        hard_indices = ranking[: CFG['loop']['hard_sample_top_k']]
        hard_samples = [fake_pool[index] for index in hard_indices]
        hard_scores = [fake_scores[index] for index in hard_indices]
        detector_model = free_gpu(detector_model)

        # STEP 3 — Rewrite hard samples with the latest agent adapter
        agent_model, agent_tokenizer = load_agent_model(agent_adapter, for_training=False)
        rewritten_samples = rewrite_batch(hard_samples, hard_scores, agent_model, agent_tokenizer)
        agent_model = free_gpu(agent_model)

        detector_model, detector_tokenizer = load_detector_model(detector_ckpt)
        rewritten_scores = score_batch(rewritten_samples, detector_model, detector_tokenizer)
        detector_model = free_gpu(detector_model)

        rewrite_frame = pd.DataFrame({
            'original_text': hard_samples,
            'original_score': hard_scores,
            'rewritten_text': rewritten_samples,
            'rewritten_score': rewritten_scores,
        })
        save_dataframe(rewrite_frame, round_dir / 'rewrites.csv')
        success_frame = rewrite_frame[rewrite_frame['rewritten_score'] < CFG['agent']['evasion_threshold']].copy()

        # STEP 4 — Retrain detector on augmented training data
        augmented_train = pd.concat(
            [
                base_train_frame,
                pd.DataFrame({
                    'title': [''] * len(success_frame),
                    'text': success_frame['rewritten_text'].tolist(),
                    'label': [1] * len(success_frame),
                }),
            ],
            ignore_index=True,
        )
        augmented_train_path = save_dataframe(augmented_train, round_dir / 'augmented_train.csv')
        current_detector_dir = detector_round_dir(round_num)
        train_detector_model(CFG, augmented_train_path, val_csv, current_detector_dir, init_checkpoint=detector_ckpt)

        # STEP 5 — Fine-tune the agent on successful rewrites
        current_agent_dir = None
        if not success_frame.empty:
            success_pairs = [
                {'original_text': row['original_text'], 'rewritten_text': row['rewritten_text']}
                for _, row in success_frame.iterrows()
            ]
            agent_model, agent_tokenizer = load_agent_model(agent_adapter, for_training=True)
            current_agent_dir = agent_round_dir(round_num)
            finetune_on_successes(success_pairs, agent_model, agent_tokenizer, current_agent_dir)
            agent_model = free_gpu(agent_model)

        # STEP 6 — Evaluate, log, and print round summary
        round_metrics, round_predictions = run_test_evaluation(current_detector_dir)
        round_metrics.update({
            'evasion_rate': compute_evasion_rate_quick(rewritten_scores, CFG['agent']['evasion_threshold']),
            'robustness_delta': round_metrics['auc'] - baseline_auc if not math.isnan(round_metrics['auc']) else math.nan,
            'successful_evasions': int(len(success_frame)),
            'detector_checkpoint': str(current_detector_dir),
            'agent_adapter': str(current_agent_dir) if current_agent_dir is not None else '',
            'status': 'completed',
        })
        log_metrics(round_metrics, round_num)
        save_dataframe(round_predictions, round_dir / 'test_predictions.csv')
        save_json({'round': round_num, **round_metrics}, round_dir / 'round_metrics.json')
        print(round_metrics)

    except Exception as exc:
        error_metrics = {'status': 'error', 'error_message': str(exc)}
        log_metrics(error_metrics, round_num)
        save_json({'round': round_num, **error_metrics}, round_dir / 'round_error.json')
        print(f'Round {round_num} failed: {exc}')
        if isinstance(exc, (ImportError, ModuleNotFoundError)):
            raise
        continue


## Cell 13 — Metrics Functions

This cell defines the standalone metrics helpers requested in the plan so the final evaluation section can run independently from the training cells if needed.


In [ ]:
def compute_evasion_rate(scores: Sequence[float], threshold: float) -> float:
    if not scores:
        return 0.0
    return float(sum(score < threshold for score in scores) / len(scores))


def compute_auc_f1(labels: Sequence[int], probabilities: Sequence[float], threshold: float = 0.5) -> dict:
    label_array = np.asarray(labels)
    prob_array = np.asarray(probabilities, dtype=float)
    predictions = (prob_array >= threshold).astype(int)
    auc = float(roc_auc_score(label_array, prob_array)) if len(np.unique(label_array)) > 1 else math.nan
    f1 = float(f1_score(label_array, predictions, zero_division=0))
    return {'auc': auc, 'f1': f1}


def compute_robustness_delta(current_auc: float, baseline_auc_value: float) -> float:
    if math.isnan(current_auc) or math.isnan(baseline_auc_value):
        return math.nan
    return float(current_auc - baseline_auc_value)


def compute_bertscore(references: Sequence[str], candidates: Sequence[str]) -> float:
    if not references or not candidates:
        return 0.0
    try:
        from bert_score import score as bert_score
        _, _, f1_values = bert_score(list(candidates), list(references), lang='en', verbose=False)
        return float(f1_values.mean().item())
    except Exception:
        overlaps = []
        for reference, candidate in zip(references, candidates):
            reference_tokens = set(reference.lower().split())
            candidate_tokens = set(candidate.lower().split())
            if not reference_tokens and not candidate_tokens:
                overlaps.append(1.0)
                continue
            overlap = len(reference_tokens & candidate_tokens)
            denom = max(len(reference_tokens | candidate_tokens), 1)
            overlaps.append(overlap / denom)
        return float(np.mean(overlaps)) if overlaps else 0.0


## Cell 14 — Results And Plots

This final cell loads the cumulative metrics CSV from Drive, plots `evasion_rate_vs_round` and `auc_f1_vs_round`, saves the figures back to Drive, prints the final results table, and shows a lightweight checkpoint summary for ablation-style comparison.


In [ ]:
metrics_csv_path = resolve_path(CFG['runtime']['metrics_csv'])
metrics_frame = load_metrics_frame()
if metrics_frame.empty:
    raise FileNotFoundError(metrics_csv_path)
save_dataframe(metrics_frame, metrics_csv_path)
metrics_frame = metrics_frame.sort_values('round').reset_index(drop=True)
completed_metrics = metrics_frame[metrics_frame['status'] == 'completed'].copy() if 'status' in metrics_frame.columns else metrics_frame.copy()
completed_metrics = completed_metrics.drop_duplicates(subset=['round'], keep='last').sort_values('round').reset_index(drop=True)

plots_dir = ensure_dir(CFG['runtime']['plots_dir'])

if not completed_metrics.empty and 'evasion_rate' in completed_metrics.columns:
    plt.figure(figsize=(8, 5))
    plt.plot(completed_metrics['round'], completed_metrics['evasion_rate'], marker='o')
    plt.title('Evasion Rate vs Round')
    plt.xlabel('Round')
    plt.ylabel('Evasion Rate')
    plt.grid(True, alpha=0.3)
    evasion_plot_path = plots_dir / 'evasion_rate_vs_round.png'
    plt.tight_layout()
    plt.savefig(evasion_plot_path, dpi=200)
    plt.show()
    plt.close()

if not completed_metrics.empty and {'auc', 'f1'}.issubset(completed_metrics.columns):
    plt.figure(figsize=(8, 5))
    plt.plot(completed_metrics['round'], completed_metrics['auc'], marker='o', label='AUC')
    plt.plot(completed_metrics['round'], completed_metrics['f1'], marker='o', label='F1')
    plt.title('AUC and F1 vs Round')
    plt.xlabel('Round')
    plt.ylabel('Metric Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    auc_f1_plot_path = plots_dir / 'auc_f1_vs_round.png'
    plt.tight_layout()
    plt.savefig(auc_f1_plot_path, dpi=200)
    plt.show()
    plt.close()

print('Final results table:')
display(completed_metrics)

checkpoint_summary = {
    'generator_checkpoint': str(generator_ckpt_dir),
    'baseline_detector': str(baseline_detector_dir),
    'detector_rounds': [str(path) for path in sorted(resolve_path(CFG['detector']['checkpoint_dir']).glob('round_*'))],
    'agent_rounds': [str(path) for path in sorted(resolve_path(CFG['agent']['checkpoint_dir']).glob('round_*')) if (path / 'adapter_config.json').exists()],
}

print('Checkpoint summary:')
print(json.dumps(checkpoint_summary, indent=2))

save_json({'completed_metrics_rows': len(completed_metrics), 'checkpoint_summary': checkpoint_summary}, Path(CFG['runtime']['artifacts_dir']) / 'final_summary.json')


## Cell 17 — Hugging Face Hub Export

Use the next cells to push the trained artifacts from Drive to Hugging Face Hub. This export path is organized as three model repos so each trained component has a clean home:

- `YOUR_OWNER/gan-vs-det-ai-gpt2-generator`
- `YOUR_OWNER/gan-vs-det-ai-biobert-detector`
- `YOUR_OWNER/gan-vs-det-ai-biogpt-agent`

`YOUR_OWNER` can be your personal username or a Hugging Face organization. The detector repo will contain the baseline plus all detector round checkpoints, and the agent repo will contain the LoRA adapters produced across rounds.


In [ ]:
from getpass import getpass

from huggingface_hub import HfApi, login


def get_hf_write_token() -> str:
    token = os.getenv('HF_WRITE_TOKEN', '').strip()
    if token:
        return token
    try:
        from google.colab import userdata
        token = (userdata.get('HF_WRITE_TOKEN') or '').strip()
        if token:
            return token
    except Exception:
        pass
    token = getpass('Paste your Hugging Face write token: ').strip()
    if not token:
        raise ValueError('A Hugging Face write token is required for uploads.')
    return token


HF_OWNER = ''
HF_PRIVATE_REPOS = False

HF_REPOS = {
    'generator': {
        'repo_id': f'{HF_OWNER}/gan-vs-det-ai-gpt2-generator' if HF_OWNER else '',
        'local_path': generator_ckpt_dir,
    },
    'detector': {
        'repo_id': f'{HF_OWNER}/gan-vs-det-ai-biobert-detector' if HF_OWNER else '',
        'local_path': resolve_path(CFG['detector']['checkpoint_dir']),
    },
    'agent': {
        'repo_id': f'{HF_OWNER}/gan-vs-det-ai-biogpt-agent' if HF_OWNER else '',
        'local_path': resolve_path(CFG['agent']['checkpoint_dir']),
    },
}

print('Recommended Hugging Face model repos:')
recommended_repo_names = {
    'generator': 'gan-vs-det-ai-gpt2-generator',
    'detector': 'gan-vs-det-ai-biobert-detector',
    'agent': 'gan-vs-det-ai-biogpt-agent',
}
for component_name, repo_spec in HF_REPOS.items():
    suggested_repo = repo_spec['repo_id'] or f'<your-owner>/{recommended_repo_names[component_name]}'
    print(f'- {component_name}: {suggested_repo}')

print('\nFill `HF_OWNER` in this cell before continuing if it is still blank.')


In [ ]:
if not HF_OWNER:
    raise ValueError('Set `HF_OWNER` in the previous cell to your Hugging Face username or org name before uploading.')

hf_write_token = get_hf_write_token()
login(token=hf_write_token, add_to_git_credential=False)
hf_api = HfApi(token=hf_write_token)
hf_profile = hf_api.whoami()

print('Authenticated Hugging Face account:')
print(json.dumps({
    'name': hf_profile.get('name'),
    'type': hf_profile.get('type'),
    'orgs': [org.get('name', org) if isinstance(org, dict) else org for org in hf_profile.get('orgs', [])],
}, indent=2))

for component_name, repo_spec in HF_REPOS.items():
    local_path = Path(repo_spec['local_path'])
    if not local_path.exists():
        raise FileNotFoundError(f'Missing local artifact directory for {component_name}: {local_path}')
    if not repo_spec['repo_id']:
        raise ValueError(f'Missing repo_id for {component_name}. Fill `HF_OWNER` in the previous cell.')
    print(f'{component_name}: {local_path} -> {repo_spec["repo_id"]}')


In [ ]:
created_repos = {}
for component_name, repo_spec in HF_REPOS.items():
    response = hf_api.create_repo(
        repo_id=repo_spec['repo_id'],
        repo_type='model',
        private=HF_PRIVATE_REPOS,
        exist_ok=True,
    )
    created_repos[component_name] = getattr(response, 'repo_id', repo_spec['repo_id'])

print('Ready repos:')
print(json.dumps(created_repos, indent=2))


In [ ]:
upload_results = {}
for component_name, repo_spec in HF_REPOS.items():
    local_path = Path(repo_spec['local_path'])
    print(f'Uploading {component_name} from {local_path} to {repo_spec["repo_id"]} ...')
    commit_info = hf_api.upload_folder(
        folder_path=str(local_path),
        repo_id=repo_spec['repo_id'],
        repo_type='model',
        commit_message=f'Upload {component_name} artifacts from Colab',
    )
    upload_results[component_name] = {
        'repo_id': repo_spec['repo_id'],
        'local_path': str(local_path),
        'commit_url': str(commit_info),
        'repo_url': f'https://huggingface.co/{repo_spec["repo_id"]}',
    }

print('Upload complete:')
print(json.dumps(upload_results, indent=2))


## Cell 18 — Load HF Models And Test Outputs

These cells download the uploaded Hugging Face repos back into Colab, resolve the latest usable checkpoint inside each repo, and run quick sanity-check inference for the generator, detector, and BioGPT agent.


In [ ]:
from huggingface_hub import snapshot_download

HF_LOAD_REPOS = {
    'generator': HF_REPOS['generator']['repo_id'] if 'HF_REPOS' in globals() else '',
    'detector': HF_REPOS['detector']['repo_id'] if 'HF_REPOS' in globals() else '',
    'agent': HF_REPOS['agent']['repo_id'] if 'HF_REPOS' in globals() else '',
}

HF_LOAD_TOKEN = globals().get('hf_write_token') or os.getenv('HF_WRITE_TOKEN', '').strip() or None
HF_TEST_TEXTS = [
    'The patient received aspirin for seven days and experienced no major adverse effects.',
    'A miracle herb cures every cancer instantly with zero side effects, doctors say.',
    'Researchers reported reduced inflammation in mice after a controlled intervention.',
]
HF_NUM_GENERATIONS = 3


def find_latest_model_subdir(root: Path, required_file: str, fallback_names: Sequence[str] = ()) -> Optional[Path]:
    candidates: List[Path] = []
    for fallback_name in fallback_names:
        candidate = root / fallback_name
        if candidate.is_dir() and (candidate / required_file).exists():
            candidates.append(candidate)
    if (root / required_file).exists():
        candidates.append(root)
    for candidate in sorted(path for path in root.glob('round_*') if path.is_dir()):
        if (candidate / required_file).exists():
            candidates.append(candidate)
    return candidates[-1] if candidates else None


hf_snapshot_dirs: dict[str, Path] = {}
for component_name, repo_id in HF_LOAD_REPOS.items():
    if not repo_id:
        raise ValueError(f'Missing Hugging Face repo id for {component_name}. Fill `HF_OWNER` or override `HF_LOAD_REPOS`.')
    hf_snapshot_dirs[component_name] = Path(snapshot_download(repo_id=repo_id, repo_type='model', token=HF_LOAD_TOKEN))

hf_resolved_paths = {
    'generator': hf_snapshot_dirs['generator'],
    'detector': find_latest_model_subdir(hf_snapshot_dirs['detector'], 'config.json', fallback_names=('baseline',)),
    'agent': find_latest_model_subdir(hf_snapshot_dirs['agent'], 'adapter_config.json'),
}

if hf_resolved_paths['detector'] is None:
    raise FileNotFoundError(f'Could not find a detector checkpoint inside {hf_snapshot_dirs["detector"]}')

print(json.dumps({key: str(value) if value is not None else None for key, value in hf_resolved_paths.items()}, indent=2))


In [ ]:
hf_generator_model, hf_generator_tokenizer, _ = load_generator_model(hf_resolved_paths['generator'])
hf_generated_samples = generate_fake_batch(HF_NUM_GENERATIONS, hf_generator_model, hf_generator_tokenizer)
hf_generator_model = free_gpu(hf_generator_model)

print('Generator samples from HF:')
for index, sample in enumerate(hf_generated_samples, start=1):
    print(f'[{index}] {sample}\n')

hf_detector_model, hf_detector_tokenizer = load_detector_model(hf_resolved_paths['detector'])
hf_detector_inputs = HF_TEST_TEXTS + hf_generated_samples
hf_detector_scores = score_batch(hf_detector_inputs, hf_detector_model, hf_detector_tokenizer)
hf_detector_model = free_gpu(hf_detector_model)

hf_detector_frame = pd.DataFrame({
    'text': hf_detector_inputs,
    'fake_probability': hf_detector_scores,
})
hf_detector_frame['predicted_label'] = (hf_detector_frame['fake_probability'] >= CFG['agent']['evasion_threshold']).astype(int)
hf_detector_frame['source'] = ['manual_test'] * len(HF_TEST_TEXTS) + ['generator_sample'] * len(hf_generated_samples)

print('Detector predictions from HF:')
display(hf_detector_frame)


In [ ]:
if hf_resolved_paths['agent'] is None:
    print('No BioGPT LoRA adapter was found in the HF agent repo yet, so the agent rewrite test is skipped.')
else:
    hf_agent_inputs = HF_TEST_TEXTS[:2]
    forced_scores = [CFG['agent']['high_conf_threshold'] + 0.1] * len(hf_agent_inputs)

    hf_agent_model, hf_agent_tokenizer = load_agent_model(hf_resolved_paths['agent'], for_training=False)
    hf_rewrites = rewrite_batch(hf_agent_inputs, forced_scores, hf_agent_model, hf_agent_tokenizer)
    hf_agent_model = free_gpu(hf_agent_model)

    hf_detector_model, hf_detector_tokenizer = load_detector_model(hf_resolved_paths['detector'])
    hf_before_scores = score_batch(hf_agent_inputs, hf_detector_model, hf_detector_tokenizer)
    hf_after_scores = score_batch(hf_rewrites, hf_detector_model, hf_detector_tokenizer)
    hf_detector_model = free_gpu(hf_detector_model)

    hf_agent_frame = pd.DataFrame({
        'original_text': hf_agent_inputs,
        'rewritten_text': hf_rewrites,
        'fake_probability_before': hf_before_scores,
        'fake_probability_after': hf_after_scores,
        'delta': [after - before for before, after in zip(hf_before_scores, hf_after_scores)],
    })

    print('Agent rewrite test from HF:')
    display(hf_agent_frame)
